In [4]:
import requests
import pandas as pd
import os
import config
from sqlalchemy import create_engine
from googleapiclient.discovery import build

In [5]:
API_KEY = config.YouTubeAPI_KEY
CHANNEL_ID = config.VAUSH_CHANNEL_ID
URL = f'https://www.googleapis.com/youtube/v3/channels?part=contentDetails&id={CHANNEL_ID}&key={API_KEY}'

response = requests.get(URL)
data = response.json()

uploads_playlist_id = data['items'][0]['contentDetails']['relatedPlaylists']['uploads']
print(f"Uploads Playlist ID: {uploads_playlist_id}")

Uploads Playlist ID: UU1E-JS8L0j1Ei70D9VEFrPQ


In [6]:
data

{'kind': 'youtube#channelListResponse',
 'etag': 'J1lqi2IrW7NGhQouQokCvOFnEM4',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': 'dpSuyUhSXYI8y3IH7vrlgkN9cAk',
   'id': 'UC1E-JS8L0j1Ei70D9VEFrPQ',
   'contentDetails': {'relatedPlaylists': {'likes': '',
     'uploads': 'UU1E-JS8L0j1Ei70D9VEFrPQ'}}}]}

In [7]:
def get_video_ids(api_key, playlist_id):
    base_url = 'https://www.googleapis.com/youtube/v3/playlistItems'
    video_ids = []
    next_page_token = None

    while True:
        url = f'{base_url}?part=contentDetails&playlistId={playlist_id}&maxResults=50&key={api_key}'
        if next_page_token:
            url += f'&pageToken={next_page_token}'

        response = requests.get(url)
        data = response.json()

        for item in data['items']:
            video_ids.append(item['contentDetails']['videoId'])

        next_page_token = data.get('nextPageToken')
        if not next_page_token:
            break

    return video_ids

API_KEY = config.YouTubeAPI_KEY
UPLOADS_PLAYLIST_ID = uploads_playlist_id

video_ids = get_video_ids(API_KEY, UPLOADS_PLAYLIST_ID)
print(f"Total Videos: {len(video_ids)}")
for video_id in video_ids:
    print(video_id)

Total Videos: 2523
osRXHIQVaek
gbKkDsDbKxE
fy3CIMMN2Bg
lnHMj84u064
sl3f9QMg0N0
LKZ-ttVn8do
DkbOpzl-zVM
2sA1YZFFQUE
AM2fmvEcuz0
8D7ifZiL010
6_DmPDPphD0
8VIhsxhd5f0
Dr5osugKc0k
WiZbR9akDp4
yHwBrcJ1Ex8
fiN6mN8XPww
-UKkNbB-DjQ
fFmvpDhNq8U
439k6LXHLxw
EVcK7NOoWkE
ICA2b_74esg
6WHXX3Kb1Ms
dxzaYbchIAU
fWMkIFEUGhE
YnN3q_IKGsE
LILCv8AOf_M
cUsKowavw04
IaN_NQMhsEY
cVTv8qvtOZM
-dnnPPGvZx4
4rJ4I-oTUd8
pkYbDLDA0Bc
X1j1YCf9qCs
CtFI5CI7Xsg
kO4pKd28rwk
inakO3nMEpo
S5tGWMCAkKE
7AQ_5izwEfY
prQf1Do7Wn8
advEkH1G92g
2879LRcfT84
fb36ZCPQ9Jk
_i7JmjBErnc
KJ-Si9afD4I
CJ0uxLE_w24
P1O8Oulm8sM
XgA5V899HZE
Zrh4spI_n1c
BD7rzhTspNk
D-uQzsnDLgw
xy1ll_23gsc
wRpgW-7ZcOA
5DlYdg5Ic6Y
TeLes_JmtvU
O4LteR-TBAY
BqpgCxwebok
C8wwbO84xGg
1_BWkwo3UiA
MhuBmxxLQ98
PO2xQPU_0Lc
RdFmEkrnuEw
nIB4I9A1Hd4
XhHVIcPn2xU
lQJJmQpMlRg
dvdurCdzuFg
6eo1neg7FL4
jzH43MFT9SA
IltcSrrh3lo
rDUaklV93iY
l2fnjOEn7Z8
YmLhdv2dHTs
zl0Q_jwOKAU
-j5d9-92HSc
aaOE2DpAaAU
ibcVRf6RXGA
3DmiYU2EYP4
dGLUqIXoAKI
ErdzMHvHx64
KLfAnWcd5I8
dbPtrNurKvo
ruDqUJ8usiw
e94dOKJc8

In [8]:
ids_df = df = pd.DataFrame(video_ids, columns=['VIDEO_ID'])
ids_df['CHANNEL'] = 'Vaush'
ids_df

,VIDEO_ID,CHANNEL
0,osRXHIQVaek,Vaush
1,gbKkDsDbKxE,Vaush
2,fy3CIMMN2Bg,Vaush
3,lnHMj84u064,Vaush
4,sl3f9QMg0N0,Vaush
...,...,...
2518,4CZUNd-N1ko,Vaush
2519,3G9x8rgrGWQ,Vaush
2520,c8yQDtLeb14,Vaush
2521,JlJkVPn2NRM,Vaush


In [9]:
ids_df['VIDEO_ID'].value_counts()

VIDEO_ID
-94STO1-D5Q    1
osRXHIQVaek    1
gbKkDsDbKxE    1
fy3CIMMN2Bg    1
lnHMj84u064    1
              ..
4rJ4I-oTUd8    1
pkYbDLDA0Bc    1
X1j1YCf9qCs    1
CtFI5CI7Xsg    1
kO4pKd28rwk    1
Name: count, Length: 2523, dtype: int64

In [11]:
# Example connection details, replace with your actual credentials
DATABASE_TYPE = config.DATABASE_TYPE
DBAPI = config.DBAPI
ENDPOINT = config.ENDPOINT
USER = config.USER
PASSWORD = config.PASSWORD
PORT = config.PORT
DATABASE = config.YOUTUBE_DATABASE

# Create the database URL
DATABASE_URL = f'{DATABASE_TYPE}+{DBAPI}://{USER}:{PASSWORD}@{ENDPOINT}:{PORT}/{DATABASE}'

# Create the engine
engine = create_engine(DATABASE_URL)

In [12]:
# Insert DataFrame into PostgreSQL table
ids_df.to_sql('YOUTUBE_IDS', engine, if_exists='append', index=False)

523

In [13]:
def get_video_details(video_ids):
    """
    Fetch details for a list of video IDs from YouTube Data API V3 and insert into PostgreSQL.
    Args:
        video_ids (list): List of YouTube video IDs.
    Returns:
        None
    """
    for i in range(0, len(video_ids), 50):  # YouTube API allows up to 50 IDs per request
        batch_ids = video_ids[i:i+50]
        request = youtube.videos().list(
            part='snippet,contentDetails,statistics',
            id=','.join(batch_ids)
        )
        response = request.execute()
        video_details = response.get('items', [])
        
        # Process the fetched video details and convert to a DataFrame
        video_data = []
        for video in video_details:
            video_info = {
                'VIDEO_ID': video['id'],
                'TITLE': video['snippet']['title'],
                'DESCRIPTION': video['snippet']['description'],
                'PUBLISHED_AT': video['snippet']['publishedAt'],
                'VIEW_COUNT': video['statistics'].get('viewCount', 0),
                'LIKE_COUNT': video['statistics'].get('likeCount', 0),
                'DISLIKE_COUNT': video['statistics'].get('dislikeCount', 0),
                'COMMENT_COUNT': video['statistics'].get('commentCount', 0),
                'DURATION': video['contentDetails']['duration']
            }
            video_data.append(video_info)
        
        # Convert to DataFrame
        video_df = pd.DataFrame(video_data)
        # Insert the DataFrame into the PostgreSQL table
        video_df.to_sql('VIDEO_STATISTICS', engine, if_exists='append', index=False)
        print(f'Inserted batch of {len(batch_ids)} records into VIDEO_STATISTICS')

query = '''
SELECT * 
FROM "YOUTUBE_IDS"
WHERE "CHANNEL" = 'Vaush'
'''

# Read the table into a DataFrame
youtube_ids_df = pd.read_sql(query, engine)

# Create a YouTube resource object
youtube = build('youtube', 'v3', developerKey=API_KEY)

# Get a list of all video IDs from the DataFrame
video_ids = df['VIDEO_ID'].tolist()

# Fetch video details and insert into PostgreSQL in batches
get_video_details(video_ids)

Inserted batch of 50 records into VIDEO_STATISTICS
Inserted batch of 50 records into VIDEO_STATISTICS
Inserted batch of 50 records into VIDEO_STATISTICS
Inserted batch of 50 records into VIDEO_STATISTICS
Inserted batch of 50 records into VIDEO_STATISTICS
Inserted batch of 50 records into VIDEO_STATISTICS


IntegrityError: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "VIDEO_STATISTICS_pkey"
DETAIL:  Key ("VIDEO_ID")=(22DAbcgPxwU) already exists.

[SQL: INSERT INTO "VIDEO_STATISTICS" ("VIDEO_ID", "TITLE", "DESCRIPTION", "PUBLISHED_AT", "VIEW_COUNT", "LIKE_COUNT", "DISLIKE_COUNT", "COMMENT_COUNT", "DURATION") VALUES (%(VIDEO_ID__0)s, %(TITLE__0)s, %(DESCRIPTION__0)s, %(PUBLISHED_AT__0)s, %(VIEW_COUNT ... 8823 characters truncated ... IEW_COUNT__49)s, %(LIKE_COUNT__49)s, %(DISLIKE_COUNT__49)s, %(COMMENT_COUNT__49)s, %(DURATION__49)s)]
[parameters: {'LIKE_COUNT__0': '5862', 'VIDEO_ID__0': '5RBTr-igmFU', 'DESCRIPTION__0': '🔴 Website  - https://www.vaush.gg/\n💵 Patreon  - https://www.patreon.com/vaush\n👕 MERCH - https://merch.whitefore.st/\n😎 JOIN PROGRESSIVE VICTORY: ht ... (512 characters truncated) ... & https://twitter.com/honeybunnbadger for the visuals, and https://twitter.com/sound_sierra for the audio! Thank you!\n\n#trump #politics #republican', 'VIE